# Setup

In [18]:
import pandas as pd
import requests

In [37]:
from io import StringIO

url = 'https://raw.githubusercontent.com/pythonhk/20260530-data-workshop/refs/heads/main/data/train_features.csv'
response = requests.get(url)
df = pd.read_csv(StringIO(response.text))

In [38]:
df

,record_id,route_id,origin_station,destination_station,district,encoded_transport,fare_hkd,distance_km,scheduled_duration_min,hour_of_day,...,session_id,ticket_reference,device_id_hash,distance_m,fare_hkd_rounded,route_group,is_peak_hour,service_note,ops_comment_code,route_label_variant
0,R00001,RT075,CentralStation,Admiralty,Central and Western,tram__local-standard__CTB;backup=CTB,NaN,NaN,NaN,18,...,S2031111535,TKT-331000,dev-c58114a0,6760,10,group-0,1,weather watch,OPS-D,Route-075
1,R00002,RT063,Kennedy Town,Admiralty,Central and Western,bus_local_standard_KMB,missing,2.01,14.0,7,...,S9969218457,TKT-890486,dev-cd6a3799,2010,20,group-0,1,maintenance escort,OPS-SYS,ROUTE 063
2,R00003,RT005,Central Station,Sha Tin,Sha Tin,Bus_LOCAL_Standard_K.M.B.,NaN,7.91,NaN,18,...,S1632419629,TKT-484610,dev-00faef30,7910,13,group-0,1,platform crowding,OPS-B,Route-005
3,R00004,RT030,Tsim Sha Tsui East,Sha Tin,Sha Tin,bus_EXPRESS_Standard_Kowloon Motor Bus,NaN,NaN,NaN,20,...,S5929138623,TKT-175440,dev-40babff7,7560,17,group-0,0,priority boarding,OPS-C,ROUTE 030
4,R00005,RT060,Tsim Sha Tsui,Causeway Bay,Yau Tsim Mong,tramexppremCTB,21.6,6.38,0.7 hrs,6,...,S4658130369,TKT-808744,dev-ba75cac0,6380,22,group-0,0,platform crowding,OPS-C,ROUTE 060
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8402,R05292,RT039,Central Station,Sha Tin,Wan Chai,bus_local_standard_K.M.B.,2500 HKD,NaN,NaN,19,...,S2057437999,TKT-675218,dev-39237a06,2410,7,group-0,1,weather watch,OPS-D,Route-039
8403,R07727,RT026,Central.Station,tst_east,Central and Western,bus_express_premium_K.M.B._v7,2500 HKD,unknown,T29,23,...,S5504502228,TKT-425070,dev-97c6b2c3,5190,23,group-0,0,platform crowding,OPS-D,Route-026
8404,R00281,RT032,Sha Tin,Kennedy Town,Tsuen Wan,bus-XHBR_local_standard_K.M.B.,missing,NaN,NaN,10,...,S3188472060,TKT-392404,dev-226ebc8f,1470,7,group-0,0,platform crowding,OPS-D,ROUTE 032
8405,R04687,RT047,Audit Hub,Audit Hub,Tsuen Wan,maintenance_shift,HK$0.5,0.5,2.0,6,...,S5996608331,TKT-373775,dev-5c83194b,900000,9000,group-0,0,test record,OPS-SYS,Route-047


# Numeric Warm-up

## fare_hkd

In [49]:
# use unique to observe the pattern
df['fare_hkd'].unique()

<StringArray>
[        nan,   'missing',      '21.6',       '0,0',    'HK$0.5', '400.0 HKD',
       '1.0',  '8.5-13.5',    '2500.0',   '1.0 HKD',
 ...
       '9,1',   'HK$16.3', '12.9-17.9',   'HK$20.3',  '9.3 fare', 'fare=10.5',
      '11,2',   '9.5 HKD',      '3.95',  '2500 HKD']
Length: 1003, dtype: str

In [75]:
# always make a copy on the original column first to prevent re-loading
# you can use the original one for cross-check & prevent re-load of data
df['fare_hkd_cleaning'] = df['fare_hkd']

In [76]:
# say found the pattern with string - suppose it should be a numeric column
# then we need to check if there's any pattern we can find
df[df['fare_hkd_cleaning'].str.contains('[a-zA-Z]', na=False)]['fare_hkd_cleaning'].unique()

<StringArray>
[  'missing',    'HK$0.5', '400.0 HKD',   '1.0 HKD',  'fare=2.0',    'HK$6.8',
   '2.0 HKD',    'HK$6.3',    'HK$9.3', '500.0 HKD',
 ...
  '18.6 HKD',  '11.0 HKD',   'HK$10.4', 'fare=15.2',   'HK$16.3',   'HK$20.3',
  '9.3 fare', 'fare=10.5',   '9.5 HKD',  '2500 HKD']
Length: 401, dtype: str

In [71]:
# first approach - do it one by one which is much safer - won't impact other data accidentally
# say I saw ' HKD' & 'fare=' & ' fare' first
pattern_needed_to_clean = [' HKD', 'fare=', ' fare'] # put the pattern you found in the above cell

for pattern in pattern_needed_to_clean:
    # set up the condition for filtering & assigning
    condition = df['fare_hkd_cleaning'].str.contains(pattern)
    
    # .loc can also be for assigning the new value to those row filtered
    # in this case, the pattern will be replace by '' to remove it
    df.loc[condition, 'fare_hkd_cleaning'] = df[condition]['fare_hkd_cleaning'].str.replace(pattern, '')
    
# after this cell finished, you can re-run the above cell
# see if there's other pattern you would like to add to the pattern_needed_to_clean until it's all done.
    
# down side: you may need to add the pattern again in future if there's some new corrupted data

In [77]:
# second approach - using regex to work on the data in batch
# directly use the regex to extra the first decimal data & replace it
# say all string related row
condition = df['fare_hkd_cleaning'].str.contains('[a-zA-Z]', na=False)

# didn't do the assign here but you can observe the cell output
df[condition]['fare_hkd_cleaning'].str.extract(r'([\d.]+)', expand=False)

# down side: you better polish the conditional filter as there's some chances to corrupt other rows 

1         NaN
13        0.5
15      400.0
23        1.0
39        NaN
        ...  
8402     2500
8403     2500
8404      NaN
8405      0.5
8406      0.5
Name: fare_hkd_cleaning, Length: 1430, dtype: str

# .. to be continued